# beta.ipynb — DeepONet 기반 3시간 후 염분도 예측

`Alpha.ipynb`에서 생성한 라벨 데이터(`tr_labeled.csv`, `val_labeled.csv`, `ts_labeled.csv`)를 사용해
**서부산낙동강교_염분도**의 **t+3 (3시간 후)** 값을 예측하는 예제입니다.

핵심 요구사항:
- 모델: Deep Operator Network(Branch + Trunk)
- 예측: 단일 호라이즌(t+3)
- Loss: `mse`, `nll`, `physics`(+가중치), 또는 사용자 정의 함수


In [ ]:
import os
import random
from dataclasses import dataclass
from typing import Callable, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


In [ ]:
@dataclass
class Config:
    # data
    lookback: int = 24
    horizon: int = 3   # 고정: 3시간 후
    target_col: str = "서부산낙동강교_염분도"
    date_col: str = "Date"

    # candidate feature columns (존재하는 컬럼만 자동 선택)
    base_features: tuple = (
        "강서낙동강교_염분도", "구포대교_염분도", "상류7h_염분도",
        "하굿둑_외수위(EL.m)", "하굿둑_내수위(EL.m)", "하굿둑_수위차",
        "month_sin", "month_cos", "doy_sin", "doy_cos",
        "tidephi_sin", "tidephi_cos", "p_w", "I_w", "K_hat_w", "beta0_hat_w"
    )

    # physics target column (있으면 physics-guided loss 가능)
    physics_col: str = "Theoretical"

    # model
    branch_hidden: int = 128
    trunk_hidden: int = 64
    latent_dim: int = 64
    dropout: float = 0.1

    # train
    batch_size: int = 128
    lr: float = 1e-3
    epochs: int = 100
    weight_decay: float = 1e-5

    # loss options: ["mse", "nll", "physics", "custom"]
    loss_mode: str = "physics"
    lambda_phy: float = 1.0
    custom_loss_fn: Optional[Callable] = None

cfg = Config()
cfg


In [ ]:
tr_path = "../tr_labeled.csv"
val_path = "../val_labeled.csv"
ts_path = "../ts_labeled.csv"

if not (os.path.exists(tr_path) and os.path.exists(val_path) and os.path.exists(ts_path)):
    raise FileNotFoundError("먼저 Alpha.ipynb를 실행해 ../tr_labeled.csv, ../val_labeled.csv, ../ts_labeled.csv 를 생성하세요.")

tr_df = pd.read_csv(tr_path)
val_df = pd.read_csv(val_path)
ts_df = pd.read_csv(ts_path)

for _df in [tr_df, val_df, ts_df]:
    _df[cfg.date_col] = pd.to_datetime(_df[cfg.date_col])
    _df.sort_values(cfg.date_col, inplace=True)
    _df.reset_index(drop=True, inplace=True)

print("train/val/test:", tr_df.shape, val_df.shape, ts_df.shape)


In [ ]:
def add_target(df: pd.DataFrame, target_col: str, horizon: int, physics_col: Optional[str] = None):
    out = df.copy()
    out[f"y_t+{horizon}"] = out[target_col].shift(-horizon)
    if physics_col is not None and physics_col in out.columns:
        out[f"phy_t+{horizon}"] = out[physics_col].shift(-horizon)
    return out


def clean_supervised_rows(df: pd.DataFrame, feature_cols, horizon: int, physics_col: Optional[str] = None):
    need = list(feature_cols) + [f"y_t+{horizon}"]
    if physics_col is not None and f"phy_t+{horizon}" in df.columns:
        # physics 타깃은 선택적이므로 결측 허용 (loss 계산 시 mask 처리)
        pass

    out = df.copy()

    # feature/label NaN, inf 제거
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=need).reset_index(drop=True)
    return out


tr_df = add_target(tr_df, cfg.target_col, cfg.horizon, cfg.physics_col)
val_df = add_target(val_df, cfg.target_col, cfg.horizon, cfg.physics_col)
ts_df = add_target(ts_df, cfg.target_col, cfg.horizon, cfg.physics_col)

feature_cols = [c for c in cfg.base_features if c in tr_df.columns]
if cfg.target_col not in feature_cols:
    feature_cols = [cfg.target_col] + feature_cols

tr_df = clean_supervised_rows(tr_df, feature_cols, cfg.horizon, cfg.physics_col)
val_df = clean_supervised_rows(val_df, feature_cols, cfg.horizon, cfg.physics_col)
ts_df = clean_supervised_rows(ts_df, feature_cols, cfg.horizon, cfg.physics_col)

print("num features:", len(feature_cols))
print(feature_cols)
print("cleaned train/val/test:", tr_df.shape, val_df.shape, ts_df.shape)


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, df, feature_cols, lookback, horizon, target_col, physics_col=None, scaler_x=None, scaler_y=None, fit=False):
        self.df = df.reset_index(drop=True).copy()
        self.feature_cols = feature_cols
        self.lookback = lookback
        self.horizon = horizon
        self.target_col = target_col
        self.physics_col = physics_col

        X = self.df[feature_cols].to_numpy(dtype=np.float32)
        y = self.df[f"y_t+{horizon}"].to_numpy(dtype=np.float32).reshape(-1, 1)

        if scaler_x is None:
            scaler_x = StandardScaler()
        if scaler_y is None:
            scaler_y = StandardScaler()

        if fit:
            X = scaler_x.fit_transform(X)
            y = scaler_y.fit_transform(y)
        else:
            X = scaler_x.transform(X)
            y = scaler_y.transform(y)

        self.scaler_x = scaler_x
        self.scaler_y = scaler_y

        self.X = X.astype(np.float32)
        self.y = y.astype(np.float32)

        if physics_col is not None and f"phy_t+{horizon}" in self.df.columns:
            phy = self.df[f"phy_t+{horizon}"].to_numpy(dtype=np.float32).reshape(-1, 1)
            phy = np.where(np.isfinite(phy), phy, np.nan)
            phy_scaled = np.full_like(phy, np.nan, dtype=np.float32)
            valid = np.isfinite(phy.squeeze())
            if valid.any():
                phy_scaled[valid] = scaler_y.transform(phy[valid])
            self.phy = phy_scaled.astype(np.float32)
        else:
            self.phy = None

        # lookback 윈도우 기준으로 실제 유효 샘플(end index)만 사용
        candidate_idx = np.arange(self.lookback - 1, len(self.df))
        valid_idx = []
        for end in candidate_idx:
            start = end - self.lookback + 1
            x_win = self.X[start:end+1]
            y_one = self.y[end]
            if np.isfinite(x_win).all() and np.isfinite(y_one).all():
                valid_idx.append(end)
        self.idx = np.array(valid_idx, dtype=int)

        dropped = len(candidate_idx) - len(self.idx)
        if dropped > 0:
            print(f"[SeqDataset] dropped invalid windows: {dropped}")

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        end = self.idx[i]
        start = end - self.lookback + 1

        x_seq = self.X[start:end+1]                      # [L, F]
        y_tgt = self.y[end].reshape(1)                   # scalar (scaled)
        tau = np.array([self.horizon / 24.0], np.float32)  # trunk input (query)

        phy = np.array([np.nan], np.float32)
        if self.phy is not None:
            phy = self.phy[end].reshape(1)

        return torch.from_numpy(x_seq), torch.from_numpy(tau), torch.from_numpy(y_tgt), torch.from_numpy(phy)


In [ ]:
train_ds = SeqDataset(tr_df, feature_cols, cfg.lookback, cfg.horizon, cfg.target_col, cfg.physics_col, fit=True)
val_ds = SeqDataset(val_df, feature_cols, cfg.lookback, cfg.horizon, cfg.target_col, cfg.physics_col,
                    scaler_x=train_ds.scaler_x, scaler_y=train_ds.scaler_y, fit=False)
test_ds = SeqDataset(ts_df, feature_cols, cfg.lookback, cfg.horizon, cfg.target_col, cfg.physics_col,
                     scaler_x=train_ds.scaler_x, scaler_y=train_ds.scaler_y, fit=False)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)

print(len(train_ds), len(val_ds), len(test_ds))


In [ ]:
class DeepONet(nn.Module):
    """
    Branch: 과거 시퀀스 x(t-L+1:t)
    Trunk : 질의 좌표(여기서는 horizon 비율 tau=3/24)
    출력  : mean, log_var (NLL 대응)
    """
    def __init__(self, n_features, lookback, branch_hidden=128, trunk_hidden=64, latent_dim=64, dropout=0.1):
        super().__init__()
        self.branch = nn.Sequential(
            nn.Linear(n_features * lookback, branch_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(branch_hidden, latent_dim),
        )
        self.trunk = nn.Sequential(
            nn.Linear(1, trunk_hidden),
            nn.GELU(),
            nn.Linear(trunk_hidden, latent_dim),
        )
        self.head_mean = nn.Linear(latent_dim, 1)
        self.head_logvar = nn.Linear(latent_dim, 1)

    def forward(self, x_seq, tau):
        b = x_seq.shape[0]
        x_flat = x_seq.reshape(b, -1)

        z_b = self.branch(x_flat)
        z_t = self.trunk(tau)
        z = z_b * z_t  # DeepONet operator interaction

        mean = self.head_mean(z)
        logvar = self.head_logvar(z).clamp(-10.0, 5.0)
        return mean, logvar


In [ ]:
def gaussian_nll(y_true, mean, logvar):
    var = torch.exp(logvar)
    return 0.5 * (logvar + (y_true - mean) ** 2 / (var + 1e-8)).mean()


def compute_loss(loss_mode, y_true, mean, logvar, phy=None, lambda_phy=1.0, custom_loss_fn=None):
    """
    loss_mode:
      - mse      : MSE만 사용
      - nll      : Gaussian NLL
      - physics  : MSE + lambda_phy * MSE(mean, phy)
      - custom   : custom_loss_fn(y_true, mean, logvar, phy) 호출
    """
    if loss_mode == "mse":
        return F.mse_loss(mean, y_true)

    if loss_mode == "nll":
        return gaussian_nll(y_true, mean, logvar)

    if loss_mode == "physics":
        data_loss = F.mse_loss(mean, y_true)
        if phy is None or torch.isnan(phy).all():
            return data_loss
        phy_mask = ~torch.isnan(phy)
        phy_loss = F.mse_loss(mean[phy_mask], phy[phy_mask]) if phy_mask.any() else 0.0
        return data_loss + lambda_phy * phy_loss

    if loss_mode == "custom":
        if custom_loss_fn is None:
            raise ValueError("loss_mode='custom'이면 cfg.custom_loss_fn을 설정해야 합니다.")
        return custom_loss_fn(y_true=y_true, mean=mean, logvar=logvar, phy=phy)

    raise ValueError(f"Unknown loss_mode: {loss_mode}")


In [ ]:
model = DeepONet(
    n_features=len(feature_cols),
    lookback=cfg.lookback,
    branch_hidden=cfg.branch_hidden,
    trunk_hidden=cfg.trunk_hidden,
    latent_dim=cfg.latent_dim,
    dropout=cfg.dropout,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)


def run_epoch(loader, train=True):
    model.train(train)
    losses = []

    for x_seq, tau, y, phy in loader:
        x_seq = x_seq.to(device)
        tau = tau.to(device)
        y = y.to(device)
        phy = phy.to(device)

        with torch.set_grad_enabled(train):
            mean, logvar = model(x_seq, tau)
            loss = compute_loss(
                cfg.loss_mode, y_true=y, mean=mean, logvar=logvar,
                phy=phy, lambda_phy=cfg.lambda_phy, custom_loss_fn=cfg.custom_loss_fn
            )
            if not torch.isfinite(loss):
                continue
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses)) if losses else np.nan

best_val = np.inf
best_state = None

for epoch in range(1, cfg.epochs + 1):
    tr_loss = run_epoch(train_loader, train=True)
    va_loss = run_epoch(val_loader, train=False)

    if np.isfinite(va_loss) and va_loss < best_val:
        best_val = va_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1:
        print(f"[{epoch:03d}/{cfg.epochs}] train={tr_loss:.5f} | val={va_loss:.5f}")

if best_state is None:
    raise RuntimeError("유효한 val loss를 얻지 못했습니다. 입력 데이터의 NaN/Inf 여부를 다시 확인하세요.")

model.load_state_dict(best_state)
print("best val loss:", best_val)


In [ ]:
@torch.no_grad()
def predict(loader):
    model.eval()
    ys, mus, sigmas = [], [], []

    for x_seq, tau, y, phy in loader:
        x_seq = x_seq.to(device)
        tau = tau.to(device)
        mean, logvar = model(x_seq, tau)

        ys.append(y.numpy())
        mus.append(mean.cpu().numpy())
        sigmas.append(np.sqrt(np.exp(logvar.cpu().numpy())))

    y = np.concatenate(ys, axis=0)
    mu = np.concatenate(mus, axis=0)
    sigma = np.concatenate(sigmas, axis=0)

    # inverse scale
    y_inv = train_ds.scaler_y.inverse_transform(y).squeeze()
    mu_inv = train_ds.scaler_y.inverse_transform(mu).squeeze()

    nan_y = int((~np.isfinite(y_inv)).sum())
    nan_mu = int((~np.isfinite(mu_inv)).sum())
    if nan_y or nan_mu:
        print(f"[predict] non-finite count -> y: {nan_y}, pred: {nan_mu}")

    return y_inv, mu_inv, sigma.squeeze()


def metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    dropped = int((~mask).sum())
    if dropped > 0:
        print(f"[metrics] NaN/Inf {dropped}개 제외 후 계산")

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    if y_true.size == 0:
        raise ValueError("유효한 샘플이 0개입니다. 예측/타깃에 NaN이 과도하게 포함되어 있습니다.")

    return {
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
        "N": int(y_true.size),
    }

val_y, val_pred, val_sigma = predict(val_loader)
ts_y, ts_pred, ts_sigma = predict(test_loader)

print("VAL:", metrics(val_y, val_pred))
print("TEST:", metrics(ts_y, ts_pred))


In [ ]:
# custom loss 예시

def custom_example_loss(y_true, mean, logvar, phy):
    nll = gaussian_nll(y_true, mean, logvar)
    l1 = F.l1_loss(mean, y_true)
    return 0.7 * nll + 0.3 * l1

# 사용법:
# cfg.loss_mode = "custom"
# cfg.custom_loss_fn = custom_example_loss
